In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.features import *
from src.volatility import *
from src.labeling import *

In [2]:
df = pd.read_csv("../data/raw/nifty50.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df = add_returns(df)
df = add_close_to_close_volatility(df, window=20)

In [3]:
hmm_df = pd.read_csv("../data/processed/hmm_regimes.csv", parse_dates=["Date"])

df = df.merge(
    hmm_df,
    on="Date",
    how="inner"
)

df = df.dropna().reset_index(drop=True)

df.head()

,Date,Open,High,Low,Close,Volume,return,vol_cc,hmm_regime
0,2007-10-16,5670.649902,5708.350098,5578.450195,5668.049805,0,-0.000414,0.288382,High
1,2007-10-17,5658.899902,5658.899902,5107.299805,5559.299805,0,-0.019186,0.308638,High
2,2007-10-18,5551.100098,5736.799805,5269.649902,5351.000000,0,-0.037469,0.331769,High
3,2007-10-19,5360.350098,5390.850098,5101.750000,5215.299805,0,-0.025360,0.350485,High
4,2007-10-22,5202.750000,5247.399902,5070.899902,5184.000000,0,-0.006002,0.348319,High


In [4]:
df = add_target_direction(df)
df = add_lagged_returns(df, lags=(1,5,10))
df = add_moving_average_features(df, windows=(5,10,20))
df = add_volatility_features(df, vol_col="vol_cc", lags=(1,))

df = df.dropna().reset_index(drop=True)

In [6]:
feature_cols = [
    "ret_lag_1","ret_lag_5","ret_lag_10",
    "ma_ratio_5","ma_ratio_10","ma_ratio_20",
    "vol_cc","vol_cc_lag_1"
]

In [7]:
split = int(len(df)*0.8)

train = df.iloc[:split].copy()
test  = df.iloc[split:].copy()

In [8]:
scaler = StandardScaler()

X_train = scaler.fit_transform(train[feature_cols])
X_test  = scaler.transform(test[feature_cols])

y_train = train["target"]
y_test  = test["target"]

logit = LogisticRegression(max_iter=1000)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)

gb = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    random_state=42
)

logit.fit(X_train, y_train)
rf.fit(X_train, y_train)
gb.fit(X_train, y_train)

test["pred_logit"] = logit.predict(X_test)
test["pred_rf"] = rf.predict(X_test)
test["pred_gb"] = gb.predict(X_test)

In [9]:
def regime_accuracy(df, pred_col):
    return df.groupby("hmm_regime").apply(
        lambda g: (g[pred_col] == g["target"]).mean()
    )

acc_logit = regime_accuracy(test, "pred_logit")
acc_rf = regime_accuracy(test, "pred_rf")
acc_gb = regime_accuracy(test, "pred_gb")

regime_results = pd.DataFrame({
    "Logistic": acc_logit,
    "RandomForest": acc_rf,
    "GradientBoost": acc_gb
})

regime_results

,Logistic,RandomForest,GradientBoost
hmm_regime,,,
High,0.613636,0.659091,0.659091
Low,0.563380,0.579812,0.575117
Medium,0.531616,0.529274,0.526932


In [10]:
regime_results["RF_gain"] = regime_results["RandomForest"] - regime_results["Logistic"]
regime_results["GB_gain"] = regime_results["GradientBoost"] - regime_results["Logistic"]

regime_results

,Logistic,RandomForest,GradientBoost,RF_gain,GB_gain
hmm_regime,,,,,
High,0.613636,0.659091,0.659091,0.045455,0.045455
Low,0.563380,0.579812,0.575117,0.016432,0.011737
Medium,0.531616,0.529274,0.526932,-0.002342,-0.004684
